# **DuckDB** Views

## Iceburg Table Connection

### Load Jar Files

In [ ]:
import os
import sys
# 1. Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

### Spark Connection

In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = r"C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

# staging_table_name = "staging.Integration.employee_Staging"
# wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    # print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)


## DuckDB Connection

In [ ]:
# import sqlite3
# import pandas as pd
import duckdb

# SQLITE_DB_PATH = r"C:\Users\progr\Downloads\WideWorldImporters.db"

con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=True)

TestSQL = """
SELECT 
city.WWI_City_ID AS "City Code",
city.City AS "City Name",
city.State_Province AS "State/Province",
city.Country AS "Country",
city.Continent AS "Continent",
city.Sales_Territory AS "Sales Territory",
city.Region AS "Region",
city.Subregion AS "Subregion",
city.Latest_Recorded_Population AS "Latest Recorded Population",
customer.WWI_Customer_ID AS "Customer Code",
customer.Customer AS "Customer Name",
customer.Bill_To_Customer AS "Billing Customer Name",
customer.Category AS "Customer Category",
customer.Buying_Group AS "Buying Customer Group",
customer.Primary_Contact AS "Primary Contact",
customer.Postal_Code AS "Postal Code",
stockitem.WWI_Stock_Item_ID AS "Stock Item Code",
stockitem.Stock_Item AS "Stock Item Name",
stockitem.Color AS "Item Color",
stockitem.Selling_Package AS "Item Selling Package",
stockitem.Buying_Package AS "Item Buying Package",
stockitem.Brand AS "ItemBrand",
stockitem.Size As "Item Size",
stockitem.Lead_Time_Days AS "Item Lead Time (Days)",
stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer",
stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock",
stockitem.Barcode AS "Item Barcode",
stockitem.Tax_Rate AS "Item Tax Rate",
stockitem.Unit_Price AS "Item Unit Price",
stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price",
stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
orderDate.ISO_Week_Number AS "Order ISO Week Number",
orderDate.Date AS "Order Date",
orderDate.Day_Number AS "Order Day Number",
orderDate.Day AS "Order Day",
orderDate.Month AS "Order Month",
orderDate.Short_Month AS "Order Short Month",
orderDate.Calendar_Month_Number AS "Order Calendar Month Number",
orderDate.Calendar_Month_Label AS "Order Calendar Month Label",
orderDate.Calendar_Year AS "Order Calendar Year",
orderDate.Calendar_Year_Label AS "Order Calendar Year Label",
orderDate.Fiscal_Month_Number AS "Order Fiscal Month Number",
orderDate.Fiscal_Month_Label AS "Order Fiscal Month Label",
orderDate.Fiscal_Year AS "Order Fiscal Year",
orderDate.Fiscal_Year_Label AS "Order Fiscal Year Label",
pickerDate.Date AS "Picker Date",
pickerDate.Day_Number AS "Picker Day Number",
pickerDate.Day AS "Picker Day",
pickerDate.Month AS "Picker Month",
pickerDate.Short_Month AS "Picker Short Month",
pickerDate.Calendar_Month_Number AS "Picker Calendar Month Number",
pickerDate.Calendar_Month_Label AS "Picker Calendar Month Label",
pickerDate.Calendar_Year AS "Picker Calendar Year",
pickerDate.Calendar_Year_Label AS "Picker Calendar Year Label",
pickerDate.Fiscal_Month_Number AS "Picker Fiscal Month Number",
pickerDate.Fiscal_Month_Label AS "Picker Fiscal Month Label",
pickerDate.Fiscal_Year AS "Picker Fiscal Year",
pickerDate.Fiscal_Year_Label AS "Picker Fiscal Year Label",
pickerDate.ISO_Week_Number AS "Picker ISO Week Number",
SalesPerson.WWI_Employee_ID AS "Sales Person Code",
SalesPerson.Employee AS "Sales Person Name",
SalesPerson.Preferred_Name AS "Sales Person Preferred Name",
SalesPerson.Is_Salesperson AS "Is Sales Person",
PickerPerson.WWI_Employee_ID AS "Picker Person Code",
PickerPerson.Employee AS "Picker Person Name",
PickerPerson.Preferred_Name AS "Picker Person Preferred Name",
PickerPerson.Is_Salesperson AS "Is Picker Person",
orders.WWI_Order_ID AS "Order Code",
orders.WWI_Backorder_ID AS "Order Backorder Code",
orders.Description AS "Order Description",
orders.Package AS "Order Package",
orders.Quantity AS "Order Quantity",
orders.Unit_Price AS "Order Unit Price",
orders.Tax_Rate AS "Order Tax Rate",
orders.Total_Excluding_Tax AS "Order Total Excluding Tax",
orders.Tax_Amount AS "Order Tax Amount",
orders.Total_Including_Tax AS "Order Total Including Tax"
FROM 
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/order', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orders LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/City', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS city ON orders.City_Key = city.City_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Customer', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS customer ON orders.Customer_Key = customer.Customer_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orderDate ON orders.Order_Date_Key = orderDate.Date LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS pickerDate ON orders.Picked_Date_Key = pickerDate.Date LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
LIMIT 10
"""

df_objects = con.execute(TestSQL).df()

display(df_objects)

# with sqlite3.connect(SQLITE_DB_PATH) as conn:
#     # 1. Read table into a Pandas DataFrame
#     df_orders = pd.read_sql_query(TestSQL, conn)
#     print(df_orders.head())
con.close()
    

## Load DataFrame

In [ ]:
df_purchase = spark.table("reporting.Fact.Purchase").alias("Purchase")
df_stock_holding = spark.table("reporting.Fact.Stock_Holding").alias("Stock_Holding")
df_order = spark.table("reporting.Fact.Order").alias("Order")
df_movement = spark.table("reporting.Fact.Movement").alias("Movement")
df_sale = spark.table("reporting.Fact.Sale").alias("Sale")
df_payment_method = spark.table("reporting.Dimension.payment_method").alias("payment_method")
df_supplier = spark.table("reporting.Dimension.supplier").alias("supplier")
df_city = spark.table("reporting.Dimension.city").alias("city")
df_stock_item = spark.table("reporting.Dimension.stock_item").alias("stock_item")
df_customer = spark.table("reporting.Dimension.customer").alias("customer")
df_date = spark.table("reporting.Dimension.date").alias("date")
df_transaction_type = spark.table("reporting.Dimension.transaction_type").alias("transaction_type")
df_employee = spark.table("reporting.Dimension.employee").alias("employee")
df_transaction = spark.table("reporting.Fact.Transaction").alias("Transaction")

### Fact df_order Analysis

In [ ]:
thesql = """
SELECT 
city.WWI_City_ID AS `City Code`,
city.City AS `City Name`,
city.State_Province AS `State/Province`,
city.Country AS `Country`,
city.Continent AS `Continent`,
city.Sales_Territory AS `Sales Territory`,
city.Region AS `Region`,
city.Subregion AS `Subregion`,
city.Latest_Recorded_Population AS `Latest Recorded Population`,
customer.WWI_Customer_ID AS `Customer Code`,
customer.Customer AS `Customer Name`,
customer.Bill_To_Customer AS `Billing Customer Name`,
customer.Category AS `Customer Category`,
customer.Buying_Group AS `Buying Customer Group`,
customer.Primary_Contact AS `Primary Contact`,
customer.Postal_Code AS `Postal Code`,
stockitem.WWI_Stock_Item_ID AS `Stock Item Code`,
stockitem.Stock_Item AS `Stock Item Name`,
stockitem.Color AS `Item Color`,
stockitem.Selling_Package AS `Item Selling Package`,
stockitem.Buying_Package AS `Item Buying Package`,
stockitem.Brand AS `ItemBrand`,
stockitem.Size As `Item Size`,
stockitem.Lead_Time_Days AS `Item Lead Time (Days)`,
stockitem.Quantity_Per_Outer AS `Item Quantity Per Outer`,
stockitem.Is_Chiller_Stock AS `Is Chiller Item Stock`,
stockitem.Barcode AS `Item Barcode`,
stockitem.Tax_Rate AS `Item Tax Rate`,
stockitem.Unit_Price AS `Item Unit Price`,
stockitem.Recommended_Retail_Price AS `Item Recommended Retail Price`,
stockitem.Typical_Weight_Per_Unit AS `Item Typical Weight Per Unit`,
orderDate.ISO_Week_Number AS `Order ISO Week Number`,
orderDate.Date AS `Order Date`,
orderDate.Day_Number AS `Order Day Number`,
orderDate.Day AS `Order Day`,
orderDate.Month AS `Order Month`,
orderDate.Short_Month AS `Order Short Month`,
orderDate.Calendar_Month_Number AS `Order Calendar Month Number`,
orderDate.Calendar_Month_Label AS `Order Calendar Month Label`,
orderDate.Calendar_Year AS `Order Calendar Year`,
orderDate.Calendar_Year_Label AS `Order Calendar Year Label`,
orderDate.Fiscal_Month_Number AS `Order Fiscal Month Number`,
orderDate.Fiscal_Month_Label AS `Order Fiscal Month Label`,
orderDate.Fiscal_Year AS `Order Fiscal Year`,
orderDate.Fiscal_Year_Label AS `Order Fiscal Year Label`,
pickerDate.Date AS `Picker Date`,
pickerDate.Day_Number AS `Picker Day Number`,
pickerDate.Day AS `Picker Day`,
pickerDate.Month AS `Picker Month`,
pickerDate.Short_Month AS `Picker Short Month`,
pickerDate.Calendar_Month_Number AS `Picker Calendar Month Number`,
pickerDate.Calendar_Month_Label AS `Picker Calendar Month Label`,
pickerDate.Calendar_Year AS `Picker Calendar Year`,
pickerDate.Calendar_Year_Label AS `Picker Calendar Year Label`,
pickerDate.Fiscal_Month_Number AS `Picker Fiscal Month Number`,
pickerDate.Fiscal_Month_Label AS `Picker Fiscal Month Label`,
pickerDate.Fiscal_Year AS `Picker Fiscal Year`,
pickerDate.Fiscal_Year_Label AS `Picker Fiscal Year Label`,
pickerDate.ISO_Week_Number AS `Picker ISO Week Number`,
SalesPerson.WWI_Employee_ID AS `Sales Person Code`,
SalesPerson.Employee AS `Sales Person Name`,
SalesPerson.Preferred_Name AS `Sales Person Preferred Name`,
SalesPerson.Is_Salesperson AS `Is Sales Person`,
PickerPerson.WWI_Employee_ID AS `Picker Person Code`,
PickerPerson.Employee AS `Picker Person Name`,
PickerPerson.Preferred_Name AS `Picker Person Preferred Name`,
PickerPerson.Is_Salesperson AS `Is Picker Person`,
orders.WWI_Order_ID AS `Order Code`,
orders.WWI_Backorder_ID AS `Order Backorder Code`,
orders.Description AS `Order Description`,
orders.Package AS `Order Package`,
orders.Quantity AS `Order Quantity`,
orders.Unit_Price AS `Order Unit Price`,
orders.Tax_Rate AS `Order Tax Rate`,
orders.Total_Excluding_Tax AS `Order Total Excluding Tax`,
orders.Tax_Amount AS `Order Tax Amount`,
orders.Total_Including_Tax AS `Order Total Including Tax`
FROM Fact.Order AS orders INNER JOIN
Dimension.City AS city ON orders.City_Key = city.City_Key INNER JOIN
Dimension.Customer AS customer ON orders.Customer_Key = customer.Customer_Key INNER JOIN
Dimension.Stock_Item AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key INNER JOIN
Dimension.Date AS orderDate ON orders.Order_Date_Key = orderDate.Date INNER JOIN
Dimension.Date AS pickerDate ON orders.Picked_Date_Key = pickerDate.Date INNER JOIN
Dimension.Employee AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key INNER JOIN
Dimension.Employee AS PickerPerson ON orders.Picker_Key = PickerPerson.Employee_Key
"""
spark.sql(thesql).show()

In [ ]:
df_order.printSchema()
df_order.filter(df_order.Picker_Key > 0).orderBy("Order_Date_Key").show(10)

In [ ]:
df_employee.filter((df_employee.Employee_Key == 11) | (df_employee.Employee_Key == 20)).show(10)

In [ ]:
df_customer.filter((df_customer.Customer_Key == 11) | (df_customer.Customer_Key == 20)).show(10, truncate=False)

In [ ]:
df_date.show(5, truncate=False)

## DuckDB Connection

### Create View SQL

#### customer_order_details

```sql
SELECT 
city.WWI_City_ID AS "City Code",
city.City AS "City Name", city.State_Province AS "State/Province", city.Country AS "Country", city.Continent AS "Continent", city.Sales_Territory AS "Sales Territory", city.Region AS "Region", city.Subregion AS "Subregion", city.Latest_Recorded_Population AS "Latest Recorded Population", 
customer.WWI_Customer_ID AS "Customer Code", customer.Customer AS "Customer Name", customer.Bill_To_Customer AS "Billing Customer Name", customer.Category AS "Customer Category", customer.Buying_Group AS "Buying Customer Group", customer.Primary_Contact AS "Primary Contact", customer.Postal_Code AS "Postal Code", stockitem.WWI_Stock_Item_ID AS "Stock Item Code", stockitem.Stock_Item AS "Stock Item Name", stockitem.Color AS "Item Color", stockitem.Selling_Package AS "Item Selling Package", stockitem.Buying_Package AS "Item Buying Package", stockitem.Brand AS "ItemBrand", stockitem.Size As "Item Size", stockitem.Lead_Time_Days AS "Item Lead Time (Days)", stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer", stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock", stockitem.Barcode AS "Item Barcode", stockitem.Tax_Rate AS "Item Tax Rate", stockitem.Unit_Price AS "Item Unit Price", stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price", stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",  orderDate.ISO_Week_Number AS "Order ISO Week Number", orderDate.Date AS "Order Date", orderDate.Day_Number AS "Order Day Number", orderDate.Day AS "Order Day", orderDate.Month AS "Order Month", orderDate.Short_Month AS "Order Short Month", orderDate.Calendar_Month_Number AS "Order Calendar Month Number", orderDate.Calendar_Month_Label AS "Order Calendar Month Label", orderDate.Calendar_Year AS "Order Calendar Year", orderDate.Calendar_Year_Label AS "Order Calendar Year Label", orderDate.Fiscal_Month_Number AS "Order Fiscal Month Number", orderDate.Fiscal_Month_Label AS "Order Fiscal Month Label", orderDate.Fiscal_Year AS "Order Fiscal Year", orderDate.Fiscal_Year_Label AS "Order Fiscal Year Label",
pickerDate.Date AS "Picker Date", pickerDate.Day_Number AS "Picker Day Number", pickerDate.Day AS "Picker Day", pickerDate.Month AS "Picker Month", pickerDate.Short_Month AS "Picker Short Month", pickerDate.Calendar_Month_Number AS "Picker Calendar Month Number", pickerDate.Calendar_Month_Label AS "Picker Calendar Month Label", pickerDate.Calendar_Year AS "Picker Calendar Year", pickerDate.Calendar_Year_Label AS "Picker Calendar Year Label", pickerDate.Fiscal_Month_Number AS "Picker Fiscal Month Number", pickerDate.Fiscal_Month_Label AS "Picker Fiscal Month Label", pickerDate.Fiscal_Year AS "Picker Fiscal Year", pickerDate.Fiscal_Year_Label AS "Picker Fiscal Year Label", pickerDate.ISO_Week_Number AS "Picker ISO Week Number", 
SalesPerson.WWI_Employee_ID AS "Sales Person Code", SalesPerson.Employee AS "Sales Person Name", SalesPerson.Preferred_Name AS "Sales Person Preferred Name", SalesPerson.Is_Salesperson AS "Is Sales Person" PickerPerson.WWI_Employee_ID AS "Picker Person Code", PickerPerson.Employee AS "Picker Person Name", PickerPerson.Preferred_Name AS "Picker Person Preferred Name", PickerPerson.Is_Salesperson AS "Is Picker Person", 
orders.WWI_Order_ID AS "Order Code", orders.WWI_Backorder_ID AS "Order Backorder Code", orders.Description AS "Order Description", orders.Package AS "Order Package", orders.Quantity AS "Order Quantity", orders.Unit_Price AS "Order Unit Price", orders.Tax_Rate AS "Order Tax Rate", orders.Total_Excluding_Tax AS "Order Total Excluding Tax", orders.Tax_Amount AS "Order Tax Amount", orders.Total_Including_Tax AS "Order Total Including Tax"
FROM Fact.Order AS orders INNER JOIN
Dimension.City AS city ON orders.City_Key = city.City_Key INNER JOIN
Dimension.Customer AS customer ON orders.Customer_Key = customer.Customer_Key INNER JOIN
Dimension.Stock_Item AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key INNER JOIN
Dimension.Date AS orderDate ON orders.Order_Date_Key = orderDate.Date INNER JOIN
Dimension.Date AS pickerDate ON orders.Picked_Date_Key = pickerDate.Date INNER JOIN
Dimension.Employee AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key INNER JOIN
Dimension.Employee AS PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
```

In [ ]:
import duckdb

factOrderView = """
SELECT 
city.WWI_City_ID AS "City Code",
city.City AS "City Name",
city.State_Province AS "State/Province",
city.Country AS "Country",
city.Continent AS "Continent",
city.Sales_Territory AS "Sales Territory",
city.Region AS "Region",
city.Subregion AS "Subregion",
city.Latest_Recorded_Population AS "Latest Recorded Population",
customer.WWI_Customer_ID AS "Customer Code",
customer.Customer AS "Customer Name",
customer.Bill_To_Customer AS "Billing Customer Name",
customer.Category AS "Customer Category",
customer.Buying_Group AS "Buying Customer Group",
customer.Primary_Contact AS "Primary Contact",
customer.Postal_Code AS "Postal Code",
stockitem.WWI_Stock_Item_ID AS "Item ID",
stockitem.Stock_Item AS "Stock Item Name",
stockitem.Color AS "Item Color",
stockitem.Selling_Package AS "Item Selling Package",
stockitem.Buying_Package AS "Item Buying Package",
stockitem.Brand AS "ItemBrand",
stockitem.Size As "Item Size",
stockitem.Lead_Time_Days AS "Item Lead Time (Days)",
stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer",
stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock",
stockitem.Barcode AS "Item Barcode",
stockitem.Tax_Rate AS "Item Tax Rate",
stockitem.Unit_Price AS "Item Unit Price",
stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price",
stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
orderDate.ISO_Week_Number AS "Order ISO Week Number",
orderDate.Date AS "Order Date",
orderDate.Day_Number AS "Order Day Number",
orderDate.Day AS "Order Day",
orderDate.Month AS "Order Month",
orderDate.Short_Month AS "Order Short Month",
orderDate.Calendar_Month_Number AS "Order Calendar Month Number",
orderDate.Calendar_Month_Label AS "Order Calendar Month Label",
orderDate.Calendar_Year AS "Order Calendar Year",
orderDate.Calendar_Year_Label AS "Order Calendar Year Label",
orderDate.Fiscal_Month_Number AS "Order Fiscal Month Number",
orderDate.Fiscal_Month_Label AS "Order Fiscal Month Label",
orderDate.Fiscal_Year AS "Order Fiscal Year",
orderDate.Fiscal_Year_Label AS "Order Fiscal Year Label",
pickerDate.Date AS "Picker Date",
pickerDate.Day_Number AS "Picker Day Number",
pickerDate.Day AS "Picker Day",
pickerDate.Month AS "Picker Month",
pickerDate.Short_Month AS "Picker Short Month",
pickerDate.Calendar_Month_Number AS "Picker Calendar Month Number",
pickerDate.Calendar_Month_Label AS "Picker Calendar Month Label",
pickerDate.Calendar_Year AS "Picker Calendar Year",
pickerDate.Calendar_Year_Label AS "Picker Calendar Year Label",
pickerDate.Fiscal_Month_Number AS "Picker Fiscal Month Number",
pickerDate.Fiscal_Month_Label AS "Picker Fiscal Month Label",
pickerDate.Fiscal_Year AS "Picker Fiscal Year",
pickerDate.Fiscal_Year_Label AS "Picker Fiscal Year Label",
pickerDate.ISO_Week_Number AS "Picker ISO Week Number",
SalesPerson.WWI_Employee_ID AS "Sales Person Code",
SalesPerson.Employee AS "Sales Person Name",
SalesPerson.Preferred_Name AS "Sales Person Preferred Name",
SalesPerson.Is_Salesperson AS "Is Sales Person",
PickerPerson.WWI_Employee_ID AS "Picker Person Code",
PickerPerson.Employee AS "Picker Person Name",
PickerPerson.Preferred_Name AS "Picker Person Preferred Name",
PickerPerson.Is_Salesperson AS "Is Picker Person",
orders.WWI_Order_ID AS "Order Code",
orders.WWI_Backorder_ID AS "Order Backorder Code",
orders.Description AS "Order Description",
orders.Package AS "Order Package",
orders.Quantity AS "Order Quantity",
orders.Unit_Price AS "Order Unit Price",
orders.Tax_Rate AS "Order Tax Rate",
orders.Total_Excluding_Tax AS "Order Total Excluding Tax",
orders.Tax_Amount AS "Order Tax Amount",
orders.Total_Including_Tax AS "Order Total Including Tax"
FROM 
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/order', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orders LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/City', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS city ON orders.City_Key = city.City_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Customer', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS customer ON orders.Customer_Key = customer.Customer_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orderDate ON orders.Order_Date_Key = orderDate.Date LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS pickerDate ON orders.Picked_Date_Key = pickerDate.Date LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
"""

# Open database in read-write mode
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Make sure the Iceberg extension is loaded
con.execute("INSTALL iceberg; LOAD iceberg;")

CreateViewSQL = f"""
CREATE OR REPLACE VIEW customer_order_details AS
{factOrderView}
"""

con.execute(CreateViewSQL)
print("✅ View 'customer_order_details' created permanently in my_warehouse.duckdb!")
con.close()

#### Purchases_Invoice

```sql
SELECT  city.WWI_City_ID AS "City Code", city.City AS "City Name", city.State_Province AS "State/Province", city.Country AS "Country", city.Continent AS "Continent", city.Sales_Territory AS "Sales Territory", city.Region AS "Region", city.Subregion AS "Subregion", city.Latest_Recorded_Population AS "Latest 
Recorded Population",customer.WWI_Customer_ID AS "Customer Code",customer.Customer AS "Customer Name",customer.Bill_To_Customer AS "Billing Customer Name",customer.Category AS "Customer Category",customer.Buying_Group AS "Buying Customer Group",customer.Primary_Contact AS "Primary Contact",customer.Postal_Code AS "Postal Code", 
stockitem.WWI_Stock_Item_ID AS "Item ID", stockitem.Stock_Item AS "Stock Item Name", stockitem.Color AS "Item Color", stockitem.Selling_Package AS "Item Selling Package", stockitem.Buying_Package AS "Item Buying Package", stockitem.Brand AS "ItemBrand", stockitem.Size As "Item Size", stockitem.Lead_Time_Days AS "Item Lead Time (Days)", stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer", stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock", stockitem.Barcode AS "Item Barcode", stockitem.Tax_Rate AS "Item Tax Rate", stockitem.Unit_Price AS "Item Unit Price", stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price", stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
orderDate.ISO_Week_Number AS "Order ISO Week Number",orderDate.Date AS "Order Date",orderDate.Day_Number AS "Order Day Number",orderDate.Day AS "Order Day",orderDate.Month AS "Order Month",orderDate.Short_Month AS "Order Short Month",orderDate.Calendar_Month_Number AS "Order Calendar Month Number",orderDate.Calendar_Month_Label AS "Order Calendar Month Label",orderDate.Calendar_Year AS "Order Calendar Year",orderDate.Calendar_Year_Label AS "Order Calendar Year Label",orderDate.Fiscal_Month_Number AS "Order Fiscal Month Number",orderDate.Fiscal_Month_Label AS "Order Fiscal Month Label",orderDate.Fiscal_Year AS "Order Fiscal Year",orderDate.Fiscal_Year_Label AS "Order Fiscal Year Label",
pickerDate.Date AS "Picker Date", pickerDate.Day_Number AS "Picker Day Number", pickerDate.Day AS "Picker Day", pickerDate.Month AS "Picker Month", pickerDate.Short_Month AS "Picker Short Month", pickerDate.Calendar_Month_Number AS "Picker Calendar Month Number", pickerDate.Calendar_Month_Label AS "Picker Calendar Month Label", pickerDate.Calendar_Year AS "Picker Calendar Year", pickerDate.Calendar_Year_Label AS "Picker Calendar Year Label", pickerDate.Fiscal_Month_Number AS "Picker Fiscal Month Number", pickerDate.Fiscal_Month_Label AS "Picker Fiscal Month Label", pickerDate.Fiscal_Year AS "Picker Fiscal Year", pickerDate.Fiscal_Year_Label AS "Picker Fiscal Year Label", pickerDate.ISO_Week_Number AS "Picker ISO Week Number",
SalesPerson.WWI_Employee_ID AS "Sales Person Code", SalesPerson.Employee AS "Sales Person Name", SalesPerson.Preferred_Name AS "Sales Person Preferred Name", SalesPerson.Is_Salesperson AS "Is Sales Person",
PickerPerson.WWI_Employee_ID AS "Picker Person Code", PickerPerson.Employee AS "Picker Person Name", PickerPerson.Preferred_Name AS "Picker Person Preferred Name", PickerPerson.Is_Salesperson AS "Is Picker Person",
orders.WWI_Order_ID AS "Order Code", orders.WWI_Backorder_ID AS "Order Backorder Code", orders.Description AS "Order Description", orders.Package AS "Order Package", orders.Quantity AS "Order Quantity", orders.Unit_Price AS "Order Unit Price", orders.Tax_Rate AS "Order Tax Rate", orders.Total_Excluding_Tax AS "Order Total Excluding Tax", orders.Tax_Amount AS "Order Tax Amount", orders.Total_Including_Tax AS "Order Total Including Tax"
FROM 
orders LEFT OUTER JOIN
city ON orders.City_Key = city.City_Key LEFT OUTER JOIN
customer ON orders.Customer_Key = customer.Customer_Key LEFT OUTER JOIN
stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key LEFT OUTER JOIN
orderDate ON orders.Order_Date_Key = orderDate.Date LEFT OUTER JOIN
pickerDate ON orders.Picked_Date_Key = pickerDate.Date LEFT OUTER JOIN
SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key LEFT OUTER JOIN
PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
```

In [ ]:
import duckdb

factPurchaseView = """
SELECT
    purchase.WWI_Purchase_Order_ID AS "Purchase Code",
    purchase_date.Date AS "Purchased  or Order Dates",
    purchase_date.Day_Number AS "Purchased  or Order Day Number",
    purchase_date."Day" AS "Purchased  or Order Day",
    purchase_date."Month" AS "Purchased  or Order Month Name",
    purchase_date.Short_Month AS "Purchased  or Order Short Month Name",
    purchase_date.Calendar_Month_Number AS "Purchased  or Order Month Number",
    purchase_date.Calendar_Month_Label AS "Purchased  or Order Calendar Month Name",
    purchase_date.Calendar_Year AS "Purchased  or Order Year Number",
    purchase_date.Calendar_Year_Label AS "Purchased  or Order Calendar Year Name",
    purchase_date.Fiscal_Month_Number AS "Purchased  or Order Financial Month Number",
    purchase_date.Fiscal_Month_Label AS "Purchased  or Order Financial Month Name",
    purchase_date.Fiscal_Year AS "Purchased  or Order Financial Year",
    purchase_date.Fiscal_Year_Label AS "Purchased  or Order Financial Year Name",
    purchase_date.ISO_Week_Number AS "Purchased  or Order Week Number",
    supplier.WWI_Supplier_ID AS "Supplier Code",
    supplier.Supplier AS "Supplier Name",
    supplier.Category AS "Supplier Category",
    supplier.Primary_Contact AS "Primary Contact",
    supplier.Supplier_Reference AS "Supplier Reference",
    supplier.Payment_Days AS "Payment Days",
    supplier.Postal_Code AS "Postal Code",
    Stock_Item.WWI_Stock_Item_ID AS "Item Code",
    Stock_Item.Stock_Item AS "Item Name",
    Stock_Item.Color AS "Item Color",
    Stock_Item.Selling_Package AS "Item Selling Package",
    Stock_Item.Buying_Package AS "Item Buying Package",
    Stock_Item.Brand AS "Item Brand",
    Stock_Item.Size AS "Item Size",
    Stock_Item.Lead_Time_Days AS "ItemLead Time Days",
    Stock_Item.Quantity_Per_Outer AS "Item Quantity Per Outer",
    Stock_Item.Is_Chiller_Stock AS "Is Chiller Item",
    Stock_Item.Barcode AS "Item Barcode",
    Stock_Item.Tax_Rate AS "Item Tax Rate",
    Stock_Item.Unit_Price AS "Item Unit Price",
    Stock_Item.Recommended_Retail_Price AS "Item Recommended Retail Price",
    Stock_Item.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
    purchase.Ordered_Outers AS "Ordered Outers",
    purchase.Ordered_Quantity AS "Ordered Quantity",
    purchase.Received_Outers AS "Received Outers",
    purchase.Package AS "Purchases Package",
    purchase.Is_Order_Finalized AS "Is Purchases Finalized"
FROM
    iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/purchase', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS purchase
LEFT JOIN iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS purchase_date ON
    ((purchase.Date_Key = purchase_date.Date))
LEFT JOIN iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/supplier', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS supplier ON
    ((purchase.Supplier_Key = supplier.Supplier_Key))
LEFT JOIN iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS Stock_Item ON
    ((purchase.Stock_Item_Key = Stock_Item.Stock_Item_Key));
"""

# Open database in read-write mode
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Make sure the Iceberg extension is loaded
con.execute("INSTALL iceberg; LOAD iceberg;")

CreateViewSQL = f"""
CREATE OR REPLACE VIEW Purchases_Invoice AS
{factOrderView}
"""

con.execute(CreateViewSQL)
print("✅ View 'Purchases_Invoice' created permanently in my_warehouse.duckdb!")
con.close()

#### Sale_Invoice

```sql
SELECT
    factSale.WWI_Invoice_ID AS "Invoice Code",
    dimension_city.WWI_City_ID AS "City Code", dimension_city.City AS "City Name", dimension_city.State_Province AS "State or Province", dimension_city.Country AS Country, dimension_city.Continent AS Continent, dimension_city.Sales_Territory AS "Sales Territory", dimension_city.Region AS Region, dimension_city.Subregion AS Subregion, dimension_city.Latest_Recorded_Population AS "Latest Recorded Population in City",
    customers.WWI_Customer_ID AS "Customer Code", customers.Customer AS "Customer Full Name", customers.Bill_To_Customer AS "Billing Customer Full Name", customers.Category AS "Customer Category", customers.Buying_Group AS "Customer Buying Group", customers.Primary_Contact AS "Customer Primary Contact", customers.Postal_Code AS "Customer Postal Code",
    cbill.WWI_Customer_ID AS "Billing Customer Code", cbill.Customer AS "Billing Customer Full Name", cbill.Bill_To_Customer AS "Billing Customer Full Name", cbill.Category AS "Billing Customer Category", cbill.Buying_Group AS "Billing Customer Buying Group", cbill.Primary_Contact AS "Billing Customer Primary Contact", cbill.Postal_Code AS "Billing Customer Postal Code",
    Stock_Item.WWI_Stock_Item_ID AS "Item Code", Stock_Item.Stock_Item AS "Item Name", Stock_Item.Color AS "Item Color", Stock_Item.Selling_Package AS "Item Selling Package", Stock_Item.Buying_Package AS "Item Buying Package", Stock_Item.Brand AS "Item Brand", Stock_Item.Size AS "Item Size", Stock_Item.Lead_Time_Days AS "ItemLead Time Days", Stock_Item.Quantity_Per_Outer AS "Item Quantity Per Outer", Stock_Item.Is_Chiller_Stock AS "Is Chiller Item", Stock_Item.Barcode AS "Item Barcode", Stock_Item.Tax_Rate AS "Item Tax Rate", Stock_Item.Unit_Price AS "Item Unit Price", Stock_Item.Recommended_Retail_Price AS "Item Recommended Retail Price", Stock_Item.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
    inv_date.Date AS "Invoice Dates", inv_date.Day_Number AS "Invoice Day Number", inv_date."Day" AS "Invoice Day", inv_date."Month" AS "Invoice Month Name", inv_date.Short_Month AS "Invoice Short Month Name", inv_date.Calendar_Month_Number AS "Invoice Month Number", inv_date.Calendar_Month_Label AS "Invoice Calendar Month Name", inv_date.Calendar_Year AS "Invoice Year Number", inv_date.Calendar_Year_Label AS "Invoice Calendar Year Name", inv_date.Fiscal_Month_Number AS "Invoice Financial Month Number", inv_date.Fiscal_Month_Label AS "Invoice Financial Month Name", inv_date.Fiscal_Year AS "Invoice Financial Year", inv_date.Fiscal_Year_Label AS "Invoice Financial Year Name", inv_date.ISO_Week_Number AS "Invoice Week Number",
    del_date.Date AS "Delivery Dates", del_date.Day_Number AS "Delivery Day Number", del_date."Day" AS "Delivery Day", del_date."Month" AS "Delivery Month Name", del_date.Short_Month AS "Delivery Short Month Name", del_date.Calendar_Month_Number AS "Delivery Month Number", del_date.Calendar_Month_Label AS "Delivery Calendar Month Name", del_date.Calendar_Year AS "Delivery Year Number", del_date.Calendar_Year_Label AS "Delivery Calendar Year Name", del_date.Fiscal_Month_Number AS "Delivery Financial Month Number", del_date.Fiscal_Month_Label AS "Delivery Financial Month Name", del_date.Fiscal_Year AS "Delivery Financial Year", del_date.Fiscal_Year_Label AS "Delivery Financial Year Name", del_date.ISO_Week_Number AS "Delivery Week Number",
    employee.Employee AS "Employee Full Name", employee.Preferred_Name AS "Employee Calling Name",
    factSale.Description AS "Billing Note Description / Annotation", factSale.Package AS "Packaging Type", factSale.Quantity AS "Sale Quantity", factSale.Unit_Price AS "Unit Price of Item", factSale.Tax_Rate AS "Tax Rate on Item", factSale.Total_Excluding_Tax AS "Total Excluding Tax on Invoice", factSale.Tax_Amount AS "Tax Amount on Item", factSale.Profit AS "Earned Profit on Invoice", factSale.Total_Including_Tax AS "Tax Amount on Invoice", factSale.Total_Dry_Items AS "Total Item can be stored in Dry Place", factSale.Total_Chiller_Items AS "Total Item need stored in cooler or child Place"
FROM    factSale LEFT JOIN 
        customers ON (factSale.Customer_Key = customers.Customer_Key) LEFT JOIN 
        cbill ON (factSale.Bill_To_Customer_Key = cbill.Customer_Key) LEFT JOIN
        dimension_city ON (factSale.city_key = dimension_city.city_key) LEFT JOIN 
        Stock_Item ON (factSale.Stock_Item_Key = Stock_Item.Stock_Item_Key) LEFT JOIN 
        employee ON (factSale.Salesperson_key = employee.Employee_Key) LEFT JOIN 
        inv_date ON (factSale.Invoice_Date_Key = inv_date.Date) LEFT JOIN 
        del_date ON (factSale.Delivery_Date_Key = del_date.Date);
```

In [ ]:
import duckdb

factSaleView = """
SELECT
    factSale.WWI_Invoice_ID AS "Invoice Code",
    dimension_city.WWI_City_ID AS "City Code",
    dimension_city.City AS "City Name",
    dimension_city.State_Province AS "State or Province",
    dimension_city.Country AS Country,
    dimension_city.Continent AS Continent,
    dimension_city.Sales_Territory AS "Sales Territory",
    dimension_city.Region AS Region,
    dimension_city.Subregion AS Subregion,
    dimension_city.Latest_Recorded_Population AS "Latest Recorded Population in City",
    customers.WWI_Customer_ID AS "Customer Code",
    customers.Customer AS "Customer Full Name",
    customers.Bill_To_Customer AS "Billing Customer Full Name",
    customers.Category AS "Customer Category",
    customers.Buying_Group AS "Customer Buying Group",
    customers.Primary_Contact AS "Customer Primary Contact",
    customers.Postal_Code AS "Customer Postal Code",
    cbill.WWI_Customer_ID AS "Billing Customer Code",
    cbill.Customer AS "Billing Customer Full Name",
    cbill.Bill_To_Customer AS "Billing Customer Full Name",
    cbill.Category AS "Billing Customer Category",
    cbill.Buying_Group AS "Billing Customer Buying Group",
    cbill.Primary_Contact AS "Billing Customer Primary Contact",
    cbill.Postal_Code AS "Billing Customer Postal Code",
    Stock_Item.WWI_Stock_Item_ID AS "Item Code",
    Stock_Item.Stock_Item AS "Item Name",
    Stock_Item.Color AS "Item Color",
    Stock_Item.Selling_Package AS "Item Selling Package",
    Stock_Item.Buying_Package AS "Item Buying Package",
    Stock_Item.Brand AS "Item Brand",
    Stock_Item.Size AS "Item Size",
    Stock_Item.Lead_Time_Days AS "ItemLead Time Days",
    Stock_Item.Quantity_Per_Outer AS "Item Quantity Per Outer",
    Stock_Item.Is_Chiller_Stock AS "Is Chiller Item",
    Stock_Item.Barcode AS "Item Barcode",
    Stock_Item.Tax_Rate AS "Item Tax Rate",
    Stock_Item.Unit_Price AS "Item Unit Price",
    Stock_Item.Recommended_Retail_Price AS "Item Recommended Retail Price",
    Stock_Item.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
    inv_date.Date AS "Invoice Dates",
    inv_date.Day_Number AS "Invoice Day Number",
    inv_date."Day" AS "Invoice Day",
    inv_date."Month" AS "Invoice Month Name",
    inv_date.Short_Month AS "Invoice Short Month Name",
    inv_date.Calendar_Month_Number AS "Invoice Month Number",
    inv_date.Calendar_Month_Label AS "Invoice Calendar Month Name",
    inv_date.Calendar_Year AS "Invoice Year Number",
    inv_date.Calendar_Year_Label AS "Invoice Calendar Year Name",
    inv_date.Fiscal_Month_Number AS "Invoice Financial Month Number",
    inv_date.Fiscal_Month_Label AS "Invoice Financial Month Name",
    inv_date.Fiscal_Year AS "Invoice Financial Year",
    inv_date.Fiscal_Year_Label AS "Invoice Financial Year Name",
    inv_date.ISO_Week_Number AS "Invoice Week Number",
    del_date.Date AS "Delivery Dates",
    del_date.Day_Number AS "Delivery Day Number",
    del_date."Day" AS "Delivery Day",
    del_date."Month" AS "Delivery Month Name",
    del_date.Short_Month AS "Delivery Short Month Name",
    del_date.Calendar_Month_Number AS "Delivery Month Number",
    del_date.Calendar_Month_Label AS "Delivery Calendar Month Name",
    del_date.Calendar_Year AS "Delivery Year Number",
    del_date.Calendar_Year_Label AS "Delivery Calendar Year Name",
    del_date.Fiscal_Month_Number AS "Delivery Financial Month Number",
    del_date.Fiscal_Month_Label AS "Delivery Financial Month Name",
    del_date.Fiscal_Year AS "Delivery Financial Year",
    del_date.Fiscal_Year_Label AS "Delivery Financial Year Name",
    del_date.ISO_Week_Number AS "Delivery Week Number",
    employee.Employee AS "Employee Full Name",
    employee.Preferred_Name AS "Employee Calling Name",
    factSale.Description AS "Billing Note Description / Annotation",
    factSale.Package AS "Packaging Type",
    factSale.Quantity AS "Sale Quantity",
    factSale.Unit_Price AS "Unit Price of Item",
    factSale.Tax_Rate AS "Tax Rate on Item",
    factSale.Total_Excluding_Tax AS "Total Excluding Tax on Invoice",
    factSale.Tax_Amount AS "Tax Amount on Item",
    factSale.Profit AS "Earned Profit on Invoice",
    factSale.Total_Including_Tax AS "Tax Amount on Invoice",
    factSale.Total_Dry_Items AS "Total Item can be stored in Dry Place",
    factSale.Total_Chiller_Items AS "Total Item need stored in cooler or child Place"
FROM
    iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/sale', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS factSale
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/customer',("version" = '?'),(allow_moved_paths = CAST('t' AS BOOLEAN))) AS customers 
ON (factSale.Customer_Key = customers.Customer_Key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/customer', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS cbill 
ON (factSale.Bill_To_Customer_Key = cbill.Customer_Key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/city', ("version" = '?'), (allow_moved_paths = CAST ('t' AS BOOLEAN))) AS dimension_city 
ON (factSale.city_key = dimension_city.city_key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS Stock_Item 
ON (factSale.Stock_Item_Key = Stock_Item.Stock_Item_Key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/employee', ("version" = '?'), (allow_moved_paths = CAST ('t' AS BOOLEAN))) AS employee 
ON (factSale.Salesperson_key = employee.Employee_Key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/date', ("version" = '?'), (allow_moved_paths = CAST ('t' AS BOOLEAN))) AS inv_date 
ON (factSale.Invoice_Date_Key = inv_date.Date)
LEFT JOIN iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/date', ("version" = '?'), (allow_moved_paths = CAST ('t' AS BOOLEAN))) AS del_date
ON (factSale.Delivery_Date_Key = del_date.Date);
"""

# Open database in read-write mode
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Make sure the Iceberg extension is loaded
con.execute("INSTALL iceberg; LOAD iceberg;")

CreateViewSQL = f"""
CREATE OR REPLACE VIEW Sales_Invoice AS
{factSaleView}
"""

con.execute(CreateViewSQL)
print("✅ View 'Sales_Invoice' created permanently in my_warehouse.duckdb!")
con.close()

### Test Views

In [44]:
# import sqlite3
# import pandas as pd
import duckdb

# SQLITE_DB_PATH = r"C:\Users\progr\Downloads\WideWorldImporters.db"

con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=True)

ViewSQL = """
SELECT 
*    
FROM customer_order_details
where "Order Date" = current_date
LIMIT 10
"""

df_objects = con.execute(ViewSQL).df()

display(df_objects)

ViewSQL = """
SELECT 
*
FROM Purchases_Invoice
LIMIT 10
"""

df_objects = con.execute(ViewSQL).df()

display(df_objects)

ViewSQL = """
SELECT 
*
FROM Sales_Invoice
LIMIT 10
"""

df_objects = con.execute(ViewSQL).df()

display(df_objects)


con.close()
    

,City Code,City Name,State/Province,Country,Continent,Sales Territory,Region,Subregion,Latest Recorded Population,Customer Code,...,Order Code,Order Backorder Code,Order Description,Order Package,Order Quantity,Order Unit Price,Order Tax Rate,Order Total Excluding Tax,Order Tax Amount,Order Total Including Tax
0,13136,Glen Park,New York,United States,North America,Mideast,Americas,Northern America,502,162,...,73577,<NA>,32 mm Double sided bubble wrap 20m,Each,80,37.00,15.0,2960.0,444.00,3404.00
1,13136,Glen Park,New York,United States,North America,Mideast,Americas,Northern America,502,162,...,73577,<NA>,USB food flash drive - fortune cookie,Each,4,4.80,15.0,19.2,2.88,22.08
2,13136,Glen Park,New York,United States,North America,Mideast,Americas,Northern America,502,162,...,73577,<NA>,Developer joke mug - there are 10 types of peo...,Each,1,13.00,15.0,13.0,1.95,14.95
3,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,73578,<NA>,Developer joke mug - fun was unexpected at thi...,Each,10,13.00,15.0,130.0,19.50,149.50
4,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,73517,73587,IT joke mug - keyboard not found … press F1 to...,Each,5,13.00,15.0,65.0,9.75,74.75
5,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,73517,73587,Furry gorilla with big eyes slippers (Black) M,Each,4,32.00,15.0,128.0,19.20,147.20
6,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,73517,73587,Tape dispenser (Black),Each,60,32.00,15.0,1920.0,288.00,2208.00
7,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,73578,<NA>,Furry animal socks (Pink) S,Pair,36,5.00,15.0,180.0,27.00,207.00
8,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,73517,73587,Halloween skull mask (Gray) S,Each,12,18.00,15.0,216.0,32.40,248.40
9,25354,Oley,Pennsylvania,United States,North America,Mideast,Americas,Northern America,1282,0,...,73508,<NA>,3 kg Courier post bag (White) 300x190x95mm,Each,175,0.66,15.0,115.5,17.33,132.83


,City Code,City Name,State/Province,Country,Continent,Sales Territory,Region,Subregion,Latest Recorded Population,Customer Code,...,Order Code,Order Backorder Code,Order Description,Order Package,Order Quantity,Order Unit Price,Order Tax Rate,Order Total Excluding Tax,Order Tax Amount,Order Total Including Tax
0,7890,Cramerton,North Carolina,United States,North America,Southeast,Americas,Northern America,4165,0,...,32336,32373,Bubblewrap dispenser (Blue) 1.5m,Each,1,240.0,15.0,240.0,36.00,276.00
1,31323,Shell Knob,Missouri,United States,North America,Plains,Americas,Northern America,1379,0,...,49873,49928,"""The Gu"" red shirt XML tag t-shirt (Black) 7XL",Each,72,18.0,15.0,1296.0,194.40,1490.40
2,20047,Lostine,Oregon,United States,North America,Far West,Americas,Northern America,213,563,...,3162,3242,Shipping carton (Brown) 457x457x457mm,Each,50,2.1,15.0,105.0,15.75,120.75
3,29490,Rosa Sánchez,Puerto Rico (US Territory),United States,North America,External,Americas,Northern America,1055,431,...,37595,37606,Superhero action jacket (Blue) L,Each,4,30.0,15.0,120.0,18.00,138.00
4,15248,Herlong,California,United States,North America,Far West,Americas,Northern America,298,420,...,65830,65855,USB food flash drive - banana,Each,8,3.2,15.0,25.6,3.84,29.44
5,34584,Tunnelhill,Pennsylvania,United States,North America,Mideast,Americas,Northern America,363,100,...,8187,8259,Plush shark slippers (Gray) M,Each,6,32.0,15.0,192.0,28.80,220.80
6,17353,Karthaus,Pennsylvania,United States,North America,Mideast,Americas,Northern America,0,476,...,28312,28351,DBA joke mug - it depends (Black),Each,5,13.0,15.0,65.0,9.75,74.75
7,4291,Brown City,Michigan,United States,North America,Great Lakes,Americas,Northern America,1325,187,...,60950,60967,USB food flash drive - pizza slice,Each,8,32.0,15.0,256.0,38.40,294.40
8,25078,Oakpark,Virginia,United States,North America,Southeast,Americas,Northern America,0,0,...,39859,39940,"""The Gu"" red shirt XML tag t-shirt (Black) L",Each,84,18.0,15.0,1512.0,226.80,1738.80
9,36513,West Frostproof,Florida,United States,North America,Southeast,Americas,Northern America,0,569,...,46011,46051,Superhero action jacket (Blue) S,Each,3,25.0,15.0,75.0,11.25,86.25


,Invoice Code,City Code,City Name,State or Province,Country,Continent,Sales Territory,Region,Subregion,Latest Recorded Population in City,...,Packaging Type,Sale Quantity,Unit Price of Item,Tax Rate on Item,Total Excluding Tax on Invoice,Tax Amount on Item,Earned Profit on Invoice,Tax Amount on Invoice,Total Item can be stored in Dry Place,Total Item need stored in cooler or child Place
0,11308,32124,South La Paloma,Texas,United States,North America,Southwest,Americas,Northern America,345,...,Each,168,4.1,15.0,688.8,103.32,352.8,792.12,168,0
1,30701,17730,Kinder,Louisiana,United States,North America,Southeast,Americas,Northern America,2477,...,Each,4,13.0,15.0,52.0,7.80,34.0,59.80,4,0
2,21379,2230,Beals,Maine,United States,North America,New England,Americas,Northern America,0,...,Packet,2,240.0,15.0,480.0,72.00,303.0,552.00,2,0
3,56962,17931,Knifley,Kentucky,United States,North America,Southeast,Americas,Northern America,0,...,Each,1,13.0,15.0,13.0,1.95,8.5,14.95,1,0
4,4088,31009,Seiling,Oklahoma,United States,North America,Southwest,Americas,Northern America,860,...,Each,9,13.0,15.0,117.0,17.55,76.5,134.55,9,0
5,27182,26300,Pastura,New Mexico,United States,North America,Southwest,Americas,Northern America,23,...,Each,8,13.0,15.0,104.0,15.60,68.0,119.60,8,0
6,23956,28032,Puerto de Luna,New Mexico,United States,North America,Southwest,Americas,Northern America,141,...,Each,6,13.0,15.0,78.0,11.70,51.0,89.70,6,0
7,19684,32887,Stonefort,Illinois,United States,North America,Great Lakes,Americas,Northern America,297,...,Each,30,42.0,15.0,1260.0,189.00,570.0,1449.00,30,0
8,24183,20642,Malott,Washington,United States,North America,Far West,Americas,Northern America,487,...,Each,8,13.0,15.0,104.0,15.60,68.0,119.60,8,0
9,61247,25653,Ortley Beach,New Jersey,United States,North America,Mideast,Americas,Northern America,1209,...,Each,108,18.0,15.0,1944.0,291.60,-108.0,2235.60,108,0
